In [2]:
import pandas as pd 
import re

In [12]:
df = pd.read_csv("direcciones.csv", sep= ";", encoding= "latin1")
df = df[["VIA_CLASE","VIA_PAR","VIA_NOMBRE","NUMERO","LATITUD","LONGITUD"]]
df

,VIA_CLASE,VIA_PAR,VIA_NOMBRE,NUMERO,LATITUD,LONGITUD
0,AUTOVÍA,NaN,A-1,10000,40°29'21.84'' N,3°40'23.56'' W
1,AUTOVÍA,NaN,A-1,10000,40°29'21.45'' N,3°40'20.75'' W
2,AUTOVÍA,NaN,A-1,11000,40°29'48.13'' N,3°39'57.53'' W
3,AUTOVÍA,NaN,A-1,11000,40°29'46.27'' N,3°39'54.87'' W
4,AUTOVÍA,NaN,A-1,12000,40°30'13.33'' N,3°39'33.27'' W
...,...,...,...,...,...,...
213798,PASEO,DEL,ZURRON,49,40°28'10.77'' N,3°35'5.92'' W
213799,PASEO,DEL,ZURRON,51,40°28'10.81'' N,3°35'3.32'' W
213800,PASEO,DEL,ZURRON,53,40°28'11.34'' N,3°35'2.07'' W
213801,PASEO,DEL,ZURRON,55,40°28'11.19'' N,3°35'1.02'' W


In [13]:
regex = r"(\d+)[°º]\s*(\d+)\'(\d+\.?\d*)\'\'\s*([NSEW])"
aux = pd.DataFrame()
aux[["la_grados","la_minutos","la_segundos","la_dir"]] = df["LATITUD"].str.extract(regex)
aux[["lo_grados","lo_minutos","lo_segundos","lo_dir"]] = df["LONGITUD"].str.extract(regex)
aux[["la_grados","la_minutos","la_segundos","lo_grados","lo_minutos","lo_segundos"]] = aux[["la_grados","la_minutos","la_segundos","lo_grados","lo_minutos","lo_segundos"]].astype("float")
aux

,la_grados,la_minutos,la_segundos,la_dir,lo_grados,lo_minutos,lo_segundos,lo_dir
0,40.0,29.0,21.84,N,3.0,40.0,23.56,W
1,40.0,29.0,21.45,N,3.0,40.0,20.75,W
2,40.0,29.0,48.13,N,3.0,39.0,57.53,W
3,40.0,29.0,46.27,N,3.0,39.0,54.87,W
4,40.0,30.0,13.33,N,3.0,39.0,33.27,W
...,...,...,...,...,...,...,...,...
213798,40.0,28.0,10.77,N,3.0,35.0,5.92,W
213799,40.0,28.0,10.81,N,3.0,35.0,3.32,W
213800,40.0,28.0,11.34,N,3.0,35.0,2.07,W
213801,40.0,28.0,11.19,N,3.0,35.0,1.02,W


In [15]:
df["LATITUD"] = aux["la_grados"] + aux["la_minutos"]/60 + aux["la_segundos"]/3600
df["LONGITUD"] = aux["lo_grados"] + aux["lo_minutos"]/60 + aux["lo_segundos"]/3600
df.loc[aux["la_dir"].isin(["W","S"]),"LATITUD"] *= -1
df.loc[aux["lo_dir"].isin(["W","S"]),"LONGITUD"] *= -1
df["VIA_PAR"]

0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
         ... 
213798    DEL
213799    DEL
213800    DEL
213801    DEL
213802    DEL
Name: VIA_PAR, Length: 213803, dtype: object

In [ ]:
df

,VIA_CLASE,VIA_PAR,VIA_NOMBRE,NUMERO,LATITUD,LONGITUD
0,AUTOV�A,NaN,A-1,10000,NaN,NaN
1,AUTOV�A,NaN,A-1,10000,NaN,NaN
2,AUTOV�A,NaN,A-1,11000,NaN,NaN
3,AUTOV�A,NaN,A-1,11000,NaN,NaN
4,AUTOV�A,NaN,A-1,12000,NaN,NaN
...,...,...,...,...,...,...
213793,PASEO,DEL,ZURRON,49,NaN,NaN
213794,PASEO,DEL,ZURRON,51,NaN,NaN
213795,PASEO,DEL,ZURRON,53,NaN,NaN
213796,PASEO,DEL,ZURRON,55,NaN,NaN


In [18]:
import excepciones as e

def limpieza_fichero() -> pd.DataFrame:
    df = pd.read_csv("direcciones.csv", sep= ";", encoding= "latin1")
    df = df[["VIA_CLASE","VIA_PAR","VIA_NOMBRE","NUMERO","LATITUD","LONGITUD"]]
    regex = re.compile(r"(\d+)[°º]\s*(\d+)\'(\d+\.?\d*)\'\'\s*([NSEW])")

    aux = pd.DataFrame()
    cols_la = ["la_grados", "la_minutos", "la_segundos", "la_dir"]
    cols_lo = ["lo_grados", "lo_minutos", "lo_segundos", "lo_dir"]

    aux[cols_la] = df["LATITUD"].str.extract(regex)
    aux[cols_lo] = df["LONGITUD"].str.extract(regex)
    aux[cols_la[:-1] + cols_lo[:-1]] = aux[cols_la[:-1] + cols_lo[:-1]].astype("float")
    df["LATITUD"] = gmd_a_grados(aux[cols_la])
    df["LONGITUD"] = gmd_a_grados(aux[cols_lo])
    return df


def gmd_a_grados(data) -> pd.Series:
    grados, minutos, segundos, direccion = data.iloc[:,0], data.iloc[:,1], data.iloc[:,2], data.iloc[:,3]
    grados_dec = grados + minutos/60 + segundos/3600
    direccion = direccion.map({"N" : 1, "E" : 1, "S" : -1, "W" : -1})
    return grados_dec * direccion


def direccion_a_coordenadas(direccion:str, df:pd.DataFrame) -> tuple:
    #1) DEFINIR EL FORMATO QUE VAMOS A ACEPTAR 
    CLASES = r"AUTOVIA|AVENIDA|CALLE|CAMINO|CARRETERA|GLORIETA|PASAJE|PASEO|PLAZA|RONDA|TRAVESIA"
    PARES = r"DE LAS|DE LOS|DE LA|DEL|DE"
    regex_via = re.compile(rf"^\s*({CLASES})\s+({PARES})?\s+(.+)\s*$")
    
    #2) NORMALIZAR LA ENTRADA DEL USUARIO
    #pasar todo a mayusculas
    direccion = direccion.upper()
    #quitar tildes
    replacements = {"Á":"A", "É":"E", "Ó":"O", "Ú":"U"}
    direccion_clean = ''.join([replacements.get(caracter, caracter) for caracter in direccion])

    #3) SEPARAR VIA Y NUMERO
    dir, num = direccion_clean.split(sep=",")

    #4) CASAR PATRON
    m = regex_via.match(dir)
    if not m: 
        raise ValueError(f"No se reconoce la estructura de la vía: '{dir}'")
    
    clase = m.group(1)
    par = (m.group(2) or '').strip() or None
    via = m.group(3)
    
    #5) ENCONTRAR EN EL DF y DEVOLVER RESULTADO 
    dir, num_raw = [p.strip() for p in direccion_clean.split(",", 1)]
    mnum = re.search(r"\d+", num_raw)
    if not mnum:
        raise ValueError(f"Número inválido en: '{num_raw}' (se esperaba un entero)")
    numero = int(mnum.group())

    par_cmp = par or ""  # en el CSV, ausencia de partícula suele ir como cadena vacía
    mask = (
        (df["VIA_CLASE"]  == clase) &
        (df["VIA_PAR"].fillna("") == par_cmp) &
        (df["VIA_NOMBRE"] == via) &
        (df["NUMERO"].astype("Int64") == numero)
        )
    
    candidatos = df.loc[mask, ["LATITUD","LONGITUD"]]
    #6) GESTION DE ERRORES
    if len(candidatos) == 0: 
        raise e.AddressNotFoundError("Calle no encontrada en la base de datos")

    lat, lon = float(candidatos.iloc[0]["LATITUD"]), float(candidatos.iloc[0]["LONGITUD"])
    return (lat, lon)




direccion = "Paseo de San Francisco de Sales, 30"
lat, lon = direccion_a_coordenadas(direccion, df)
print(lat, lon)





40.44161944444444 -3.7136861111111115
